<a href="https://colab.research.google.com/github/aguaguagugugugugu/DEG/blob/main/air_llm/examples/run_all_types_of_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# install airllm

In [13]:
!pip install --upgrade pip
!pip install "torch>=2.4" --index-url https://download.pytorch.org/whl/cu121
!pip install "transformers==4.44.2"
!pip install "tokenizers==0.19.1"
!pip install "accelerate>=1.0"
!pip install safetensors huggingface-hub scipy tqdm
!pip install "bitsandbytes==0.44.1"
!pip install airllm

Looking in indexes: https://download.pytorch.org/whl/cu121
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 29.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.7 MB/s  0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> tokenizers
  Using cached tokenizers-0.19.1.tar.gz (321 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backen

# test Platypus2

In [14]:
from airllm import AutoModel

MAX_LENGTH = 128
# could use hugging face model repo id:
model = AutoModel.from_pretrained("deepseek-ai/DeepSeek-V2-Lite", profiling_mode=True)
#model = AirLLMLlama2("garage-bAInd/Platypus2-7B", profiling_mode=False)

# or use model's local path...
#model = AirLLMLlama2("/home/ubuntu/.cache/huggingface/hub/models--garage-bAInd--Platypus2-70B-instruct/snapshots/b585e74bcaae02e52665d9ac6d23f4d0dbc81a0f")

input_text = [
        #'What is the capital of China?',
        'I like',
    ]

input_tokens = model.tokenizer(input_text,
    return_tensors="pt",
    return_attention_mask=False,
    truncation=True,
    max_length=MAX_LENGTH,
    #padding=True
    )

generation_output = model.generate(
    input_tokens['input_ids'].cuda(),
    max_new_tokens=3,
    use_cache=True,
    return_dict_in_generate=True)

model.tokenizer.decode(generation_output.sequences[0])

using generic AirLLM streaming model for architecture: DeepseekV2ForCausalLM


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

found_layers:{'model.embed_tokens.': True, 'model.layers.0.': True, 'model.layers.1.': True, 'model.layers.2.': True, 'model.layers.3.': True, 'model.layers.4.': True, 'model.layers.5.': True, 'model.layers.6.': True, 'model.norm.': True, 'lm_head.': True, 'model.layers.10.': True, 'model.layers.11.': True, 'model.layers.12.': True, 'model.layers.13.': True, 'model.layers.7.': True, 'model.layers.8.': True, 'model.layers.9.': True, 'model.layers.14.': True, 'model.layers.15.': True, 'model.layers.16.': True, 'model.layers.17.': True, 'model.layers.18.': True, 'model.layers.19.': True, 'model.layers.20.': True, 'model.layers.21.': True, 'model.layers.22.': True, 'model.layers.23.': True, 'model.layers.24.': True, 'model.layers.25.': True, 'model.layers.26.': True}
saved layers already found in /root/.cache/huggingface/hub/models--deepseek-ai--DeepSeek-V2-Lite/snapshots/604d5664dddd88a0433dbae533b7fe9472482de0/splitted_model
patched: using path = /root/.cache/huggingface/hub/models--deep

RecursionError: maximum recursion depth exceeded

In [10]:
import airllm.airllm_base as ab
from transformers import AutoConfig, AutoModelForCausalLM

BaseClass = ab.AirLLMBaseModel

# Патч токенизатора
def _patched_get_tokenizer(self, hf_token=None):
    return _tok

BaseClass.get_tokenizer = _patched_get_tokenizer

_orig = BaseClass._instantiate_on_meta

def _patched(self, attn_implementation="eager"):
    # AirLLM хранит путь в self.model_local_path, а не в model_local_path_or_repo_id
    path = getattr(self, "model_local_path", None) or getattr(self, "model_local_path_or_repo_id", None)
    print("patched: using path =", path)
    try:
        cfg = AutoConfig.from_pretrained(path, trust_remote_code=True)
        return AutoModelForCausalLM.from_config(
            cfg,
            trust_remote_code=True,
            torch_dtype=self.dtype,
        )
    except Exception as e:
        print("patched _instantiate_on_meta failed:", type(e).__name__, e)
        return _orig(self, attn_implementation)

BaseClass._instantiate_on_meta = _patched
ab.bitsandbytes_installed = True
print("patched with correct path attribute")

patched with correct path attribute


In [8]:
import airllm.airllm_base as ab
from transformers import AutoConfig, AutoModelForCausalLM

# ВАЖНО: правильное имя класса — AirLLMBaseModel
BaseClass = ab.AirLLMBaseModel

def _patched_get_tokenizer(self, hf_token=None):
    return _tok

BaseClass.get_tokenizer = _patched_get_tokenizer

_orig = BaseClass._instantiate_on_meta

def _patched(self, attn_implementation="eager"):
    try:
        cfg = AutoConfig.from_pretrained(
            self.model_local_path_or_repo_id,
            trust_remote_code=True,
        )
        return AutoModelForCausalLM.from_config(
            cfg,
            trust_remote_code=True,
            torch_dtype=self.dtype,
        )
    except Exception:
        return _orig(self, attn_implementation)

BaseClass._instantiate_on_meta = _patched
ab.bitsandbytes_installed = True
print("patched get_tokenizer + _instantiate_on_meta (on AirLLMBaseModel)")

patched get_tokenizer + _instantiate_on_meta (on AirLLMBaseModel)
